# 06 — Run MILAN on the trained InceptionV3 (extension)
Same two steps as notebook 03, but for InceptionV3 (`--arch inception_v3`):
1. **Dissection** — top-k activating images for every unit in the Inception probe layers `Conv2d_2b_3x3, Mixed_5d, Mixed_6a, Mixed_6e, Mixed_7c` (~3,936 units total, ~4x ResNet18).
2. **Captioning** — feed the exemplars through the pretrained MILAN decoder (`base`) for a description per unit.

Inputs run at **299×299** (Inception's native size) — handled automatically from the arch registry. Note this stage is ~4x heavier than ResNet18; use `mps` if available, and the exemplar stage caches per layer so it resumes if interrupted.

In [1]:
import os, sys, torch
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / 'milan'))
os.environ.setdefault('MILAN_DATA_DIR', str(Path.cwd().parent / 'data'))
os.environ.setdefault('MILAN_MODELS_DIR', str(Path.cwd().parent / 'models'))
os.environ.setdefault('MILAN_RESULTS_DIR', str(Path.cwd().parent / 'results'))

ARCH = 'inception_v3'
version_dir = Path(os.environ['MILAN_DATA_DIR']) / 'imagenet-spurious-text' / '50pct'
ckpt = Path(os.environ['MILAN_MODELS_DIR']) / f'{ARCH}_spurious.pth'
dissect_dir = Path(os.environ['MILAN_RESULTS_DIR']) / 'edit' / 'imagenet-spurious-text' / f'{ARCH}_spurious-50pct'
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
device

'mps'

In [2]:
# Dissection: image_size defaults to 299 for inception_v3 (from the arch registry).
from milan_repro.milan_glue.run_exemplars import run as run_exemplars
run_exemplars(version_dir, ckpt, dissect_dir, arch=ARCH, device=device)

/Users/tsaiyuntong/Documents/homeworks/dsc291/DSC291_Research_Reproduction/milan_repro/milan_glue/register.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = to

[Conv2d_2b_3x3] cached, skipping
[Mixed_5d] cached, skipping
[Mixed_6a] cached, skipping
[Mixed_6e] cached, skipping
[Mixed_7c] cached, skipping


PosixPath('/Users/tsaiyuntong/Documents/homeworks/dsc291/DSC291_Research_Reproduction/results/edit/imagenet-spurious-text/inception_v3_spurious-50pct')

In [ ]:
# Captioning is architecture-agnostic; just point it at the Inception dissect dir.
from milan_repro.milan_glue.run_descriptions import run as run_descriptions
out_csv = Path(os.environ['MILAN_RESULTS_DIR']) / f'descriptions_{ARCH}.csv'
run_descriptions(dissect_dir, out_csv, device=device)

load imagenet-spurious-text/inception_v3_spurious-50pct:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
from milan_repro.editing.identify_text_neurons import annotate
annotate(out_csv, out_csv.with_name(f'descriptions_{ARCH}_annotated.csv'))

In [ ]:
# Peek + text-neuron fraction per layer (the layer-depth analysis for Inception).
import pandas as pd
df = pd.read_csv(out_csv.with_name(f'descriptions_{ARCH}_annotated.csv'))
print('text neurons:', int(df['is_text_neuron'].sum()), '/', len(df))
display(df.groupby('layer')['is_text_neuron'].mean().rename('text_fraction'))
df[df['is_text_neuron']].head(10)